# M9 Verification: Structured Associations

M9 reports exploratory descriptive associations with explicit strata and leave-one-out diagnostics.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import display
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'paper_v9').is_dir(): ROOT=ROOT.parent
METRICS=ROOT/'paper_v9'/'data'/'metrics'
FIGURES=ROOT/'paper_v9'/'figures'; FIGURES.mkdir(parents=True,exist_ok=True)
magnitude=pd.read_csv(METRICS/'m9_planning_vs_rework_m6a_m8a.csv')
ratio=pd.read_csv(METRICS/'m9_planning_vs_rework_m6a_m8b_eligible_stratum.csv')
outcomes=pd.read_csv(METRICS/'m9_planning_vs_outcomes_m6a_t3.csv')
m6b=pd.read_csv(METRICS/'m9_planning_vs_outcomes_m6b_t3_if_approved.csv')
loo=pd.read_csv(METRICS/'m9_leave_one_out_intervals.csv')
metadata=json.loads((METRICS/'m9_structured_associations.metadata.json').read_text())

## Verification

M9 uses team-semester grain, stratifies M8b by baseline eligibility, excludes M7, and keeps M6b unavailable pending human review.

In [ ]:
paper=(ROOT/'paper_v8'/'latex_code'/'main.tex').read_text(encoding='utf-8')
assert paper.index(r'\subsubsection{RQ3:') < paper.index(r'\textbf{M9 --')
assert metadata['rq']=='RQ3'
assert metadata['coverage']=={'team_semesters':14,'baseline_eligible':12,'m6b_observed':9,'leave_one_out_rows':52}
assert metadata['m7_used_as_predictor'] is False
assert metadata['m6b_status']=='approved_human_review'
assert len(magnitude)==2 and len(ratio)==2 and len(outcomes)==8 and len(m6b)==30 and len(loo)==52
assert m6b['analysis_id'].eq('m6b_structured_content').all()
assert loo['n_remaining'].ge(3).all()
print('M9 contract, strata, approved M6b, and influence checks: PASS')

## Traceability and artifact demo

The official association outputs preserve separate constructs and strata.

In [ ]:
traceability=pd.DataFrame([
 {'v8_recommendation':'Associate planning with clean rework magnitude.','v9_decision':'Use M6a predictors and M8a across 14 team-semesters.','status':'applied','evidence':'M9a CSV','limitation_or_approval':'Descriptive small n.'},
 {'v8_recommendation':'Stratify rework ratio by baseline eligibility.','v9_decision':'Use M8b only among 12 eligible cases.','status':'applied','evidence':'M9b CSV','limitation_or_approval':'Path provenance eligibility.'},
 {'v8_recommendation':'Include T3 outcomes and influence diagnostics.','v9_decision':'Publish M9c, approved M6b structured associations, and leave-one-out rows; exclude M7 and preserve missingness.','status':'applied','evidence':'M9c/M9d/LOO CSVs and metadata','limitation_or_approval':'M6b fields remain exploratory and non-composite.'}
])
assert set(traceability['status'])=={'applied'}
display(traceability)
display(magnitude); display(ratio); display(outcomes); display(m6b.head(10)); display(loo.head(8))
plot=pd.concat([magnitude.assign(family='clean_rework_magnitude'),ratio.assign(family='clean_rework_ratio'),outcomes.assign(family='evaluator_outcomes'),m6b.assign(family='m6b_structured_content')],ignore_index=True)
plot['label']=plot['predictor']+' / '+plot['outcome']
figure=px.bar(plot,x='label',y='spearman_rho',color='analysis_id',title='M9 exploratory Spearman associations')
figure.update_layout(xaxis_tickangle=-45)
figure.write_html(METRICS/'m9_structured_associations.html',include_plotlyjs='cdn')
for ext in ('pdf','svg','png'): figure.write_image(FIGURES/f'm9_structured_associations.{ext}',scale=2 if ext=='png' else 1)
for ext in ('pdf','svg','png'): assert (FIGURES/f'm9_structured_associations.{ext}').is_file() and (FIGURES/f'm9_structured_associations.{ext}').stat().st_size>0
print('M9 official artifact demo and article-ready figures generated: PASS')

## Preliminary RQ3 analysis

M9 is exploratory and descriptive. Small denominators, influential cases, missingness, and pending M6b review prohibit causal or confirmatory claims.

In [ ]:
summary=pd.DataFrame([{'team_semesters':metadata['coverage']['team_semesters'],'baseline_eligible':metadata['coverage']['baseline_eligible'],'association_rows':len(magnitude)+len(ratio)+len(outcomes),'leave_one_out_rows':len(loo),'analysis_level':metadata['unit_of_analysis'],'inference':metadata['inference'],'m6b_status':metadata['m6b_status'],'m7_used_as_predictor':metadata['m7_used_as_predictor']}])
assert summary['inference'].eq('exploratory_descriptive').all()
assert summary['m7_used_as_predictor'].eq(False).all()
display(summary)
print('Preliminary RQ3 reading: no causal or confirmatory claim is supported.')